In [2]:
import re
def clean_lyrics(text):
    # 1. Chuyển hết xuống dòng thành space
    text = text.replace('\r\n', ' ').replace('\n', ' ')
    
    # 2. Loại bỏ các tag như [ĐK:], [RAP:], [Bridge:], [x 4]...
    text = re.sub(r'\[.*?\]', '', text)          # bỏ [ĐK:], [RAP:], ...
    text = re.sub(r'x \d+', '', text)            # bỏ x 4, x 2, ...

    # 3. Loại bỏ ký tự không phải chữ, số hoặc dấu câu cơ bản
    text = re.sub(r'[^a-zA-Z0-9À-ỹ.,!? ]+', '', text)

    # 4. Chuyển nhiều khoảng trắng thành 1 khoảng trắng
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [3]:
import json
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn import tree
import matplotlib.pyplot as plt

# -----------------------
# 1. Load dataset JSON
# -----------------------
with open("dataset.json", "r", encoding="utf-8") as f:
    data= json.load(f)


In [4]:
for item in data:
    item['noi_dung'] = clean_lyrics(item['noi_dung'])

In [5]:

df = pd.DataFrame(data)
print(df)

df['text'] = df['ten_bai_hat'] + " " + df['noi_dung'] + " " + df['nghe_si'] 



               ten_bai_hat                                           noi_dung  \
0                    Em Bé  Bae bae, em tự dưng muốn đi gym hằng ngày Bae ...   
1        hai mươi hai (22)  Cuộc đời này là màu hồng mẹ nói lúc con ra đời...   
2   trời giấu trời mang đi                Lời bài hát sẽ sớm được cập nhật...   
3            ưng quá chừng  Sao hôm nay, lại cứ ngẩn ngơ thế này Sao mà bâ...   
4            Santa Tell Me  Bài hát Santa Tell Me Ariana Grande Santa tell...   
..                     ...                                                ...   
77         Cornelia Street  We were in the backseat Drunk on something str...   
78                    Mine  Ah, ah, ah Ah, ah, ah You were in college work...   
79              Sparks Fly  The way you move is like a fallen rainstorm An...   
80        Back To December  Im so glad you made time to see me Hows life? ...   
81               Speak Now  I am not the kind of girl Who should be rudely...   

          nghe_si  
0      

In [6]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Giả sử df có cột 'text' (mo_ta + noi_dung) và 'ten_bai_hat'
model = SentenceTransformer('all-MiniLM-L6-v2')

In [7]:

embeddings = model.encode(df['text'].tolist(), convert_to_numpy=True)

In [13]:
def search_song(user_text, top_k=5):
    user_emb = model.encode([user_text], convert_to_numpy=True)
    sims = cosine_similarity(user_emb, embeddings)
    top_idx = sims[0].argsort()[::-1][:top_k]

    results = [
        {
            "ten_bai_hat": df.iloc[i]['ten_bai_hat'],
            "similarity": round(float(sims[0][i]), 2)  # làm tròn 2 chữ số
        }
        for i in top_idx
    ]
    return results

query = "khi nao lay chong"
query = query.replace("\n", " ")

top_songs = search_song(query, top_k=5)
for song in top_songs:
    print(song)


{'ten_bai_hat': 'Bao Giờ Lấy Chồng', 'similarity': 0.54}
{'ten_bai_hat': 'Mùa hè', 'similarity': 0.49}
{'ten_bai_hat': 'Mùa hè', 'similarity': 0.47}
{'ten_bai_hat': 'Tình đắng như ly cà phê', 'similarity': 0.47}
{'ten_bai_hat': 'Tình đắng như ly cà phê', 'similarity': 0.46}


In [9]:
search_song(query, top_k=5)

,ten_bai_hat,text,similarity
20,Bao Giờ Lấy Chồng,Bao Giờ Lấy Chồng Năm mới lại đến em vẫn lẻ bó...,0.537017
37,Mùa hè,Mùa hè Đây là lời bài hát test cho bài ID 6.Nơ...,0.488573
39,Mùa hè,Mùa hè Đây là lời bài hát test cho bài ID 6.Nơ...,0.472170
40,Tình đắng như ly cà phê,Tình đắng như ly cà phê Đây là lời bài hát tes...,0.468820
38,Tình đắng như ly cà phê,Tình đắng như ly cà phê Đây là lời bài hát tes...,0.462493
